# Bloomberg API Examples
Connection management and data retrieval for Reference, Historical, Intraday, Streaming and BQL, powered by the xbbg 1.4.1 engine. Results are Polars by default; pass `legacy=True` for pandas. Construct with `host` and `port` to switch transparently from the local engine to a remote `serve` peer (see Remote Mode at the end).

In [ ]:
from datetime import date, datetime

from Library.Bloomberg import BloombergAPI

# Reference Data
Point-in-time descriptive and market data.

### fetch (BDP)
`fetch` returns point-in-time values in long format (`ticker`, `field`, `value`). The `with` block auto-connects and disconnects, and both arguments accept a single value or a list.

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.reference.fetch(["XBTUSD Curncy", "ETHUSD Curncy"], ["PX_LAST", "NAME"])
    display(df)

### bulk (BDS)
`bulk` fetches array/bulk fields that return a whole table per security (e.g. index members, dividend history). Rows are tagged with `ticker` and `field` plus the field's own columns.

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.reference.bulk("SPX Index", "INDX_MEMBERS")
    display(df)

### Manual Connection Management
Connect and disconnect explicitly to hold the session across independent operations.

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.reference.fetch("XBTUSD Curncy", "PX_LAST")
    display(df)
finally:
    bbg.disconnect()

# Historical Data
End-of-day time series across a date range.

### Date Ranges & Timeframes
`fetch` returns long format (`ticker`, `date`, `field`, `value`). Use `timeframe` for the periodicity (DAILY, WEEKLY, MONTHLY, ...).

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.historical.fetch("XBTUSD Curncy", "PX_LAST", start=date(2024, 1, 1), stop=date(2024, 3, 31))
    display(df)

### Weekly Periodicity
Manual connection with a weekly timeframe.

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.historical.fetch("ETHUSD Curncy", "PX_LAST", start=date(2024, 1, 1), stop=date(2024, 6, 30), timeframe="WEEKLY")
    display(df)
finally:
    bbg.disconnect()

# Intraday Data
Granular intraday bars or discrete ticks.

### Intraday Bars (single date)
`bars` mirrors xbbg's `bdib`: one date via `dt`, aggregated into `interval`-minute bars. `typ` picks the event (TRADE, BID, ASK).

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.intraday.bars("XBTUSD Curncy", dt=date(2024, 6, 14), interval=60, typ="BID")
    display(df)

### Intraday Ticks (datetime range)
`ticks` mirrors xbbg's `bdtick`: a `start`/`stop` datetime range of individual ticks.

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.intraday.ticks("XBTUSD Curncy", start=datetime(2024, 6, 14, 9), stop=datetime(2024, 6, 14, 17), event_types="BID")
    display(df)
finally:
    bbg.disconnect()

# Streaming Data
Real-time market data subscriptions.

### Streaming with Callbacks
`subscribe` dispatches each live update batch to your `callback` as a DataFrame (Polars, or pandas with `legacy=True`). Set `limit` to stop after N updates.

In [ ]:
def on_data(df):
    display(df)

with BloombergAPI() as bbg:
    bbg.streaming.subscribe("XBTUSD Curncy", "LAST_PRICE", callback=on_data, limit=2)

### Manual Streaming Lifecycle
Disconnect cleanly to drop the subscription.

In [ ]:
def on_data(df):
    display(df)

bbg = BloombergAPI().connect()
try:
    bbg.streaming.subscribe("ETHUSD Curncy", "BID", callback=on_data, limit=2)
finally:
    bbg.disconnect()

# BQL Queries
Execute Bloomberg Query Language strings via `query.execute`.

> **Note:** Requires a BQL-enabled Bloomberg subscription.

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.query.execute("get(px_last) for(['XBTUSD Curncy'])")
    display(df)

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.query.execute("get(px_last) for(['XBTUSD Curncy'])")
    display(df)
finally:
    bbg.disconnect()

# Remote Mode (Server / Client)
The same interface works across machines: run `serve` next to a logged-in Bloomberg Terminal, then construct the API with `host` and `port` anywhere else (only `pyzmq` and `polars` are needed - no xbbg, no Bloomberg runtime). Calls travel as JSON envelopes over ZMQ REQ/REP and frames come back as Arrow IPC bytes; `legacy` is resolved client-side. On startup the server logs `Serve Operation: Listening (tcp://<host>:<port>)` - that is the address to plug into the client. Use `whitelist`/`blacklist` to filter clients by address. Streaming is local-only for now.

### Server (Terminal PC - blocking loop)

In [ ]:
BloombergAPI().serve(port=5555, token="secret", whitelist=["10.0.0.7"])

### Client (any machine that reaches the server)

In [ ]:
with BloombergAPI(host="terminal-pc", port=5555, token="secret", timeout=300) as bbg:
    df = bbg.reference.fetch(["XBTUSD Curncy", "ETHUSD Curncy"], ["PX_LAST", "NAME"])
    display(df)